# 02. Наивный Байес — классификация

## Только классификация

Наивный Байес — семейство вероятностных классификаторов, основанных на **теореме Байеса**.

Название состоит из двух частей:

- **Байес** — потому что используется теорема Байеса;
- **Наивный** — потому что вводится сильное упрощающее предположение о независимости признаков при условии класса.

Разберём это предположение, формулу алгоритма, варианты модели, гиперпараметры, достоинства и недостатки.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")

## 1. Теорема Байеса

Пусть `C` — класс объекта, а `X` — его признаки.

Теорема Байеса:

$$
P(C\mid X)=\frac{P(X\mid C)P(C)}{P(X)}.
$$

Здесь:

- `P(C|X)` — насколько вероятен класс после того, как мы увидели признаки;
- `P(X|C)` — насколько вероятны такие признаки при данном классе;
- `P(C)` — априорная вероятность класса;
- `P(X)` — общая вероятность наблюдаемых признаков.

Для классификации нам достаточно сравнивать значения для разных классов, поэтому общий знаменатель можно не вычислять.

## 2. Почему алгоритм называется «наивным»?

Представим объект с тремя признаками:

$$
X=(x_1,x_2,x_3).
$$

В общем случае признаки могут зависеть друг от друга. Наивный Байес делает упрощающее предположение:

$$
P(x_1,x_2,x_3\mid C)
=
P(x_1\mid C)P(x_2\mid C)P(x_3\mid C).
$$

То есть признаки считаются **условно независимыми при фиксированном классе**.

Это сильное предположение может быть неправдоподобным в реальных данных — отсюда слово **«наивный»**.

Интересно, что даже при нарушении этого предположения классификатор часто способен показывать хорошее качество.

## 3. Как работает классификация

Для каждого класса вычисляется величина, пропорциональная:

$$
P(C_k)\prod_j P(x_j\mid C_k).
$$

Затем выбирается класс с максимальным значением:

$$
\hat C=\arg\max_{C_k}P(C_k)\prod_jP(x_j\mid C_k).
$$

На практике вместо произведения вероятностей часто используют логарифмы:

$$
\log P(C_k)+\sum_j\log P(x_j\mid C_k),
$$

потому что сумма численно удобнее произведения большого количества маленьких чисел.

## 4. Варианты Naive Bayes

Выбор варианта зависит от природы признаков.

### GaussianNB

Предполагается, что непрерывный признак внутри каждого класса имеет нормальное распределение.

Подходит, например, для числовых измерений.

### BernoulliNB

Используется для бинарных признаков — например, признак присутствует/отсутствует.

### MultinomialNB

Особенно часто применяется для неотрицательных счётчиков, например частот слов в текстовой классификации.

В этом ноутбуке для двумерной визуализации используем `GaussianNB`.

In [ ]:
X, y = make_classification(
    n_samples=300,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=42
)

plt.figure(figsize=(8,6))
plt.scatter(X[y==0,0], X[y==0,1], label="Класс 0")
plt.scatter(X[y==1,0], X[y==1,1], label="Класс 1")
plt.xlabel("X1")
plt.ylabel("X2")
plt.title("Данные для Naive Bayes")
plt.legend()
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

model = GaussianNB()
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

## 5. Визуализация границы классификации

На двумерной плоскости можно увидеть, какие области алгоритм относит к каждому классу.

In [ ]:
x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
y_min, y_max = X[:,1].min()-1, X[:,1].max()+1
xx, yy = np.meshgrid(
    np.linspace(x_min,x_max,400),
    np.linspace(y_min,y_max,400)
)

grid = np.c_[xx.ravel(), yy.ravel()]
z = model.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8,6))
plt.contourf(xx, yy, z, alpha=0.20)
plt.scatter(X[y==0,0], X[y==0,1], label="Класс 0", edgecolor="black")
plt.scatter(X[y==1,0], X[y==1,1], label="Класс 1", edgecolor="black")
plt.xlabel("X1")
plt.ylabel("X2")
plt.title("Граница классификации Gaussian Naive Bayes")
plt.legend()
plt.show()

## 6. Псевдоалгоритм Gaussian Naive Bayes

Упрощённая схема обучения:

1. Для каждого класса посчитать его априорную вероятность.
2. Для каждого признака и каждого класса оценить среднее.
3. Для каждого признака и класса оценить дисперсию.
4. Для нового объекта вычислить условную вероятность каждого признака.
5. Перемножить условные вероятности и умножить на вероятность класса.
6. Выбрать класс с максимальным результатом.

In [ ]:
def pseudo_gaussian_nb_predict(X_train, y_train, new_object):
    classes = np.unique(y_train)
    scores = {}

    for c in classes:
        X_c = X_train[y_train == c]

        prior = len(X_c) / len(X_train)
        mean = X_c.mean(axis=0)
        variance = X_c.var(axis=0) + 1e-9

        # Плотность нормального распределения по каждому признаку
        likelihoods = (
            1 / np.sqrt(2 * np.pi * variance)
            * np.exp(-(new_object - mean)**2 / (2 * variance))
        )

        # Наивное предположение: перемножаем вклады признаков
        scores[c] = prior * np.prod(likelihoods)

    return max(scores, key=scores.get)

## 7. Гиперпараметры

У разных вариантов Naive Bayes параметры отличаются.

### GaussianNB

- `var_smoothing` — стабилизирующая добавка к дисперсиям; помогает избежать проблем с очень маленькими дисперсиями.

### BernoulliNB

- `alpha` — сглаживание;
- `binarize` — порог преобразования признаков в бинарные.

### MultinomialNB

- `alpha` — параметр сглаживания;
- `fit_prior` — учитывать ли априорные вероятности классов;
- `class_prior` — можно задать априорные вероятности вручную.

Важно: в Naive Bayes гиперпараметров обычно меньше, чем у многих сложных алгоритмов, и это одна из причин его популярности.

## 8. Зачем нужно сглаживание?

Если некоторый признак никогда не встречался в обучающих объектах определённого класса, его оценённая вероятность может оказаться равной нулю.

При перемножении вероятностей это способно обнулить весь результат для класса.

**Сглаживание** не позволяет вероятностям становиться проблемно равными нулю и делает оценку устойчивее.

## 9. Достоинства

- Простая математическая идея.
- Быстрое обучение и предсказание.
- Хорошо работает на небольших объёмах данных.
- Может хорошо работать в задачах классификации текста.
- Требует относительно мало гиперпараметров.
- Естественно работает с вероятностной интерпретацией классов.

## 10. Недостатки

- Основное ограничение — предположение об условной независимости признаков.
- Сильно коррелированные признаки могут нарушать исходную модель.
- Неправильный выбор распределения для признаков может ухудшить результат.
- Простая вероятностная модель может проигрывать более гибким алгоритмам на сложных границах.
- Вероятности не всегда хорошо откалиброваны, даже если классификация получается качественной.

### Главная идея

Naive Bayes жертвует реалистичностью предположений ради **простоты, скорости и устойчивости**. Это хороший пример того, как достаточно сильное упрощение иногда даёт полезный классификатор.